# Pyr/CA3 Mesh Download And Decimation By List

This notebook generalizes the proven single-root mesh workflow from `02_pyr_mesh_download.ipynb` to a root-ID batch. It uses flat `seg_m195` as the primary mesh source and lazily uses the CAVE-advertised Graphene source only when the flat source raises `MeshDecodeError` with `Manifest not found`. Existing outputs are validated and reused, new H5/PLY outputs are written through validated temporary files, results are logged incrementally, per-root failures are isolated, and final copy/paste-ready ID lists are printed. `missing_manifest_ids` means IDs still unavailable after the Graphene fallback attempt.


## Parameters

A list containing one root ID should behave like the proven one-off workflow. Set `overwrite_existing = True` manually only when replacing existing H5 or PLY files is intentional.

In [1]:
root_ids = [648518346458354357]
# [648518346342322415, 648518346342517767, 648518346342658823, 648518346342659591, 648518346343910304, 648518346344436265, 648518346344453929, 648518346344454185, 648518346344476201, 648518346344561766, 648518346345548728, 648518346345567160, 648518346346706831, 648518346359197539, 648518346359399367, 648518346359402339, 648518346359408327, 648518346359408583, 648518346359430499, 648518346359450567, 648518346359451847, 648518346359453639, 648518346359471715, 648518346359472483, 648518346359478115, 648518346359483747, 648518346359484615, 648518346359496035, 648518346359503815, 648518346359505351, 648518346359512263, 648518346359518563, 648518346359547079, 648518346359555427, 648518346359831651, 648518346359865220, 648518346360067050, 648518346360095630, 648518346360104846, 648518346360663989, 648518346360733365, 648518346360733621, 648518346360736949, 648518346360743861, 648518346362144525, 648518346362201844, 648518346362203892, 648518346362209780, 648518346362214644, 648518346362222068, 648518346362251789, 648518346362255629, 648518346362267636, 648518346362267892, 648518346362276084, 648518346362289652, 648518346362302196, 648518346362329588, 648518346362545820, 648518346362609675, 648518346362615563, 648518346362618635, 648518346362648333, 648518346362653965, 648518346362671117, 648518346362707612, 648518346362708380, 648518346362799260, 648518346362991971, 648518346363075228, 648518346364501457, 648518346364501969, 648518346364505297, 648518346364605393, 648518346364959631, 648518346364962191, 648518346365006991, 648518346365039898, 648518346365085978, 648518346365087514, 648518346365087770, 648518346365088794, 648518346365094667, 648518346365094923, 648518346365142042, 648518346365145626, 648518346365148186, 648518346365148954, 648518346365151002, 648518346365152538, 648518346365152794, 648518346365153562, 648518346365154330, 648518346365157914, 648518346365159706, 648518346365160218, 648518346365160474, 648518346365161498, 648518346365162266, 648518346365165082, 648518346365167642, 648518346365169946, 648518346365171482, 648518346365175066, 648518346365177626, 648518346365179162, 648518346365188634, 648518346365191706, 648518346365194010, 648518346365197338, 648518346365200922, 648518346365205786, 648518346365206810, 648518346365213210, 648518346365215002, 648518346365216794, 648518346365217306, 648518346365217562, 648518346365219098, 648518346365219354, 648518346365220122, 648518346365225498, 648518346365226266, 648518346365226522, 648518346365228826, 648518346365229082, 648518346365232154, 648518346365238810, 648518346365239066, 648518346365330517, 648518346365331285, 648518346365334613, 648518346365339221, 648518346365341269, 648518346365343317, 648518346365344341, 648518346365345365, 648518346365345621, 648518346365364821, 648518346365633877, 648518346365635413, 648518346365635925, 648518346365637205, 648518346365637973, 648518346365638741, 648518346365638997, 648518346365643861, 648518346365644629, 648518346365644885, 648518346365645653, 648518346365652053, 648518346365655381, 648518346365660757, 648518346365663317, 648518346365663573, 648518346365664853, 648518346365668693, 648518346365671765, 648518346365672277, 648518346365678677, 648518346365680213, 648518346365680469, 648518346365702229, 648518346365702485, 648518346365704533, 648518346365704789, 648518346365761818, 648518346365763098, 648518346365765146, 648518346365765658, 648518346365766426, 648518346365768218, 648518346366263893, 648518346366371413, 648518346366373973, 648518346366374229, 648518346366374485, 648518346366377301, 648518346366686722, 648518346367113759, 648518346367142175, 648518346367202562, 648518346367205890, 648518346367211010, 648518346367212034, 648518346367228674, 648518346367423519, 648518346367432479, 648518346367433759, 648518346367435551, 648518346367620610, 648518346367622658, 648518346367623682, 648518346367899935, 648518346367901215, 648518346367916319, 648518346368021506, 648518346368021762, 648518346368317116, 648518346368317372, 648518346368318908, 648518346368539324, 648518346368550844, 648518346368598204, 648518346368603068, 648518346368607932, 648518346368609724, 648518346368616892, 648518346368617660, 648518346368621244, 648518346368631996, 648518346368637372, 648518346368644284, 648518346368646844, 648518346368649404, 648518346368652732, 648518346368657340, 648518346368658364, 648518346368661948, 648518346368665532, 648518346368666812, 648518346368667068, 648518346368669884, 648518346368670652, 648518346368671932, 648518346368672956, 648518346368673212, 648518346368676284, 648518346368677308, 648518346368681404, 648518346368855393, 648518346368896609, 648518346368971964, 648518346368974268, 648518346368974524, 648518346368976060, 648518346368976316, 648518346368977340, 648518346370906303, 648518346372541501, 648518346372544829, 648518346372545597, 648518346372546365, 648518346372547901, 648518346372561981, 648518346372562237, 648518346372614973, 648518346372616765, 648518346372620349, 648518346372624701, 648518346372629053, 648518346372632637, 648518346372668989, 648518346372671805, 648518346372674109, 648518346372683325, 648518346372686653, 648518346373824802, 648518346373828898, 648518346373882914, 648518346373887778, 648518346373914146, 648518346373914402, 648518346373919266, 648518346373921314, 648518346373922850, 648518346373929506, 648518346373934370, 648518346373939490, 648518346374112546, 648518346374718097, 648518346374729105, 648518346374732177, 648518346374735249, 648518346375898264, 648518346375900056, 648518346375911320, 648518346376593695, 648518346376616479, 648518346376631583, 648518346376684757, 648518346376693205, 648518346377555260, 648518346377565244, 648518346377567036, 648518346377994556, 648518346377995068, 648518346380340534, 648518346380454454, 648518346380468534, 648518346380535350, 648518346385674505, 648518346385717257, 648518346385858569, 648518346387278751, 648518346387295391, 648518346387306399, 648518346387306655, 648518346387318431, 648518346387337631, 648518346387988133, 648518346387996325, 648518346387997861, 648518346387998885, 648518346388004773, 648518346388006821, 648518346388008357, 648518346388017829, 648518346388027301, 648518346388052389, 648518346389251320, 648518346389274104, 648518346389292536, 648518346389296376, 648518346389297656, 648518346389315064, 648518346389580733, 648518346389583805, 648518346389584829, 648518346389599165, 648518346390883428, 648518346390898532, 648518346390901092, 648518346390901860, 648518346390923108, 648518346390923364, 648518346390923876, 648518346390924388, 648518346390925156, 648518346390926436, 648518346390927716, 648518346390928996, 648518346391196051, 648518346391197587, 648518346391198355, 648518346391198867, 648518346391249555, 648518346391250067, 648518346391255187, 648518346391262611, 648518346391274899, 648518346391276947, 648518346391277971, 648518346391278227, 648518346391288467, 648518346391419027, 648518346391419539, 648518346391426451, 648518346391431059, 648518346391437203, 648518346391437971, 648518346391440275, 648518346391441555, 648518346391443091, 648518346391665130, 648518346391665386, 648518346391665642, 648518346391667434, 648518346391667690, 648518346391667946, 648518346391674090, 648518346391674346, 648518346391674602, 648518346391675882, 648518346391838186, 648518346391838698, 648518346391868138, 648518346391877098, 648518346391878378, 648518346391878890, 648518346391882218, 648518346391884522, 648518346391886570, 648518346392873033, 648518346392874313, 648518346392937545, 648518346392939081, 648518346392943433, 648518346392959561, 648518346392973641, 648518346392975177, 648518346392976457, 648518346392976713, 648518346392976969, 648518346392977225, 648518346392977481, 648518346392979017, 648518346392979529, 648518346392993865, 648518346392994633, 648518346392996169, 648518346393059145, 648518346393059401, 648518346393060425, 648518346393072713, 648518346396855540, 648518346396856052, 648518346396856820, 648518346396862452, 648518346396902644, 648518346396938740, 648518346397316557, 648518346397319373, 648518346397362381, 648518346397429697, 648518346397450433, 648518346397457345, 648518346397458113, 648518346397470145, 648518346397480385, 648518346397481153, 648518346397483713, 648518346398378994, 648518346398407922, 648518346398409970, 648518346398451954, 648518346398455538, 648518346398866880, 648518346398868160, 648518346399048384, 648518346399148992, 648518346399149504, 648518346399165632, 648518346399166656, 648518346399166912, 648518346399365732, 648518346399368036, 648518346399395428, 648518346399401316, 648518346399406436, 648518346399483748, 648518346399762793, 648518346399785833, 648518346399786601, 648518346399929845, 648518346399938805, 648518346399941877, 648518346399976693, 648518346399977717, 648518346399981045, 648518346399983349, 648518346399984629, 648518346399985141, 648518346399987189, 648518346399990773, 648518346399991029, 648518346399994101, 648518346400073717, 648518346400093173, 648518346400094453, 648518346400098037, 648518346400263869, 648518346400743454, 648518346400804894, 648518346400947998, 648518346400949278, 648518346401002857, 648518346401024361, 648518346401024873, 648518346401067625, 648518346401094249, 648518346401095273, 648518346401101929, 648518346401180009, 648518346401183593, 648518346401202281, 648518346403341640, 648518346403971149, 648518346403976269, 648518346403981133, 648518346403982157, 648518346403983181, 648518346403984717, 648518346403985485, 648518346403988301, 648518346403990861, 648518346403991629, 648518346403991885, 648518346403992653, 648518346404011085, 648518346404012109, 648518346404012621, 648518346404013645, 648518346404022093, 648518346404022349, 648518346404022605, 648518346404035917, 648518346404036173, 648518346404040013, 648518346404040269, 648518346404042573, 648518346404453160, 648518346404454952, 648518346404455720, 648518346404460840, 648518346404461096, 648518346404465192, 648518346404467496, 648518346404468008, 648518346404475176, 648518346404477992, 648518346404478248, 648518346404483880, 648518346404484136, 648518346404520488, 648518346404523560, 648518346404555560, 648518346404580714, 648518346404606570, 648518346404643434, 648518346405290122, 648518346405321610, 648518346406375562, 648518346406376586, 648518346406420494, 648518346406420750, 648518346406525264, 648518346406792655, 648518346406795983, 648518346407847491, 648518346407847747, 648518346407855171, 648518346407856707, 648518346407856963, 648518346407867459, 648518346407868995, 648518346407871299, 648518346407871811, 648518346407873603, 648518346407909431, 648518346407910711, 648518346408028113, 648518346408029137, 648518346408029905, 648518346408031185, 648518346408055875, 648518346408058691, 648518346408058947, 648518346408063555, 648518346408064835, 648518346408110545, 648518346408110801, 648518346408111057, 648518346408114385, 648518346408117457, 648518346408118993, 648518346408120017, 648518346408120529, 648518346408121809, 648518346408124113, 648518346408124881, 648518346408127953, 648518346408130001, 648518346408141777, 648518346408156369, 648518346408158417, 648518346408164305, 648518346408170961, 648518346408595361, 648518346408596897, 648518346408597921, 648518346408598177, 648518346408600993, 648518346408628385, 648518346408630433, 648518346408630945, 648518346408633505, 648518346408634017, 648518346408635297, 648518346408873377, 648518346408874145, 648518346409743970, 648518346409744738, 648518346409745762, 648518346409753186, 648518346409757538, 648518346409757794, 648518346409760098, 648518346409761890, 648518346409762146, 648518346409765218, 648518346409766754, 648518346409769826, 648518346409920603, 648518346410021195, 648518346410021707, 648518346410022731, 648518346410039627, 648518346410039883, 648518346410040395, 648518346410044235, 648518346410044491, 648518346410046283, 648518346410048075, 648518346410051147, 648518346410201419, 648518346410405493, 648518346410405749, 648518346410410357, 648518346410411381, 648518346410412411, 648518346410417781, 648518346410421621, 648518346410425717, 648518346410430581, 648518346410431093, 648518346410433141, 648518346410435189, 648518346410438517, 648518346410440309, 648518346410442357, 648518346410460409, 648518346410461177, 648518346410461689, 648518346410465529, 648518346410467445, 648518346410469493, 648518346410469749, 648518346410472309, 648518346410476917, 648518346410478453, 648518346410479477, 648518346410479989, 648518346410481525, 648518346410482293, 648518346410483061, 648518346410483317, 648518346410483829, 648518346410484085, 648518346410484597, 648518346410485109, 648518346410485365, 648518346410487157, 648518346410487413, 648518346410487669, 648518346410487925, 648518346410488181, 648518346410490229, 648518346410490485, 648518346410491253, 648518346410491765, 648518346410492021, 648518346410492277, 648518346410492533, 648518346410492789, 648518346410494069, 648518346410616187, 648518346410624891, 648518346410631035, 648518346410648699, 648518346410678651, 648518346410689659, 648518346410733049, 648518346410733561, 648518346410735097, 648518346410738169, 648518346410740985, 648518346410742009, 648518346410745337, 648518346410753273, 648518346410753667, 648518346410753923, 648518346410763513, 648518346410763769, 648518346410765049, 648518346410765305, 648518346410765817, 648518346410767353, 648518346410767609, 648518346410768889, 648518346410772985, 648518346410781177, 648518346410804347, 648518346410816454, 648518346410816505, 648518346410819526, 648518346410822342, 648518346410822598, 648518346410825158, 648518346410825414, 648518346410826182, 648518346410827206, 648518346410827462, 648518346410837702, 648518346410844358, 648518346410844870, 648518346410845638, 648518346410847174, 648518346411077081, 648518346411077849, 648518346411078361, 648518346411078617, 648518346411092934, 648518346411101571, 648518346411102595, 648518346411102851, 648518346411145091, 648518346411149955, 648518346411150467, 648518346411170009, 648518346411171033, 648518346411171289, 648518346411171801, 648518346411172569, 648518346411182041, 648518346411182553, 648518346411182809, 648518346411183833, 648518346411184089, 648518346411184345, 648518346411185350, 648518346411185625, 648518346411185881, 648518346411186118, 648518346411186137, 648518346411186649, 648518346411189721, 648518346411191001, 648518346411191257, 648518346411207366, 648518346411209158, 648518346411216771, 648518346411219331, 648518346411221123, 648518346411221379, 648518346411223939, 648518346411224195, 648518346411226310, 648518346411226566, 648518346411227334, 648518346411228102, 648518346411228870, 648518346411229638, 648518346411230662, 648518346411230918, 648518346411231430, 648518346411249094, 648518346411249350, 648518346411250118, 648518346411250886, 648518346411316678, 648518346411317446, 648518346411317702, 648518346411319494, 648518346411343046, 648518346411343302, 648518346411344070, 648518346411345094, 648518346411345862, 648518346411482310, 648518346411482566, 648518346411482822, 648518346411483846, 648518346411484102, 648518346411484614, 648518346411486406, 648518346411486662, 648518346411486918, 648518346411542058, 648518346411571346, 648518346411572370, 648518346411573394, 648518346411573650, 648518346411574162, 648518346411575698, 648518346411575954, 648518346411576466, 648518346411576978, 648518346411577234, 648518346411578258, 648518346411589418, 648518346411627306, 648518346411643690, 648518346411648298, 648518346411687466, 648518346411700266, 648518346411703850, 648518346412104495, 648518346412105007, 648518346412105263, 648518346412106287, 648518346412108591, 648518346412114735, 648518346412115503, 648518346412118831, 648518346412121903, 648518346412127279, 648518346412127535, 648518346412154671, 648518346412155695, 648518346412157231, 648518346412158255, 648518346412159023, 648518346412160559, 648518346412160815, 648518346412161071, 648518346412162863, 648518346412163375, 648518346412166703, 648518346412168495, 648518346412170031, 648518346412494915, 648518346412505923, 648518346412751939, 648518346412752707, 648518346412753987, 648518346412755779, 648518346412756291, 648518346412757827, 648518346412758851, 648518346412766275, 648518346412766787, 648518346412771432, 648518346412787523, 648518346412787779, 648518346412788035, 648518346412788803, 648518346412795203, 648518346412803139, 648518346412807528, 648518346412813416, 648518346412819304, 648518346412828995, 648518346412830531, 648518346412831811, 648518346412845616, 648518346412857448, 648518346412886120, 648518346412894056, 648518346412897896, 648518346412922984, 648518346412932456, 648518346414129549, 648518346414131085, 648518346414135949, 648518346414148749, 648518346414149261, 648518346414152845, 648518346414154637, 648518346414168461, 648518346414185613, 648518346414186893, 648518346414187149, 648518346414188685, 648518346414189709, 648518346414190733, 648518346414194829, 648518346414195085, 648518346414196365, 648518346414196621, 648518346414197389, 648518346414198925, 648518346414200717, 648518346414201229, 648518346414215875, 648518346414216899, 648518346414217155, 648518346414217667, 648518346414217923, 648518346414388049, 648518346414397521, 648518346414525581, 648518346414526605, 648518346414526861, 648518346414528141, 648518346414529933, 648518346414530701, 648518346414550455, 648518346414551735, 648518346414552759, 648518346414555575, 648518346414559927, 648518346414560951, 648518346414561463, 648518346414561975, 648518346414569869, 648518346414570381, 648518346414570637, 648518346414571917, 648518346414579639, 648518346414582199, 648518346414584247, 648518346414585015, 648518346414585527, 648518346414586295, 648518346414588087, 648518346414594743, 648518346414596791, 648518346414597815, 648518346414598071, 648518346414598327, 648518346414601143, 648518346414610103, 648518346414624439, 648518346414700882, 648518346414720594, 648518346414723666, 648518346414760274, 648518346414772306, 648518346414800722, 648518346414815314, 648518346414817362, 648518346414854482, 648518346414860882, 648518346414862418, 648518346414884690, 648518346414885970, 648518346414895954, 648518346414896466, 648518346415024721, 648518346415031633, 648518346415034193, 648518346415095479, 648518346415097783, 648518346415202897, 648518346415216465, 648518346415216721, 648518346415216977, 648518346415220817, 648518346415223121, 648518346415313042, 648518346416989864, 648518346417007784, 648518346417454575, 648518346417455855, 648518346417467375, 648518346417467631, 648518346417469423, 648518346417471727, 648518346417482991, 648518346417484015, 648518346417485039, 648518346417486831, 648518346417487087, 648518346417489391, 648518346417649647, 648518346417650671, 648518346417652975, 648518346417656815, 648518346417659375, 648518346418571595, 648518346419429029, 648518346419429541, 648518346419640520, 648518346419646664, 648518346419672741, 648518346419673253, 648518346419673509, 648518346419676837, 648518346419679653, 648518346419680165, 648518346419680677, 648518346419680933, 648518346419681445, 648518346419684261, 648518346419781029, 648518346419793317, 648518346419794853, 648518346419797413, 648518346419797669, 648518346419799461, 648518346419884232, 648518346419929288, 648518346419929544, 648518346420051656, 648518346420068552, 648518346420074118, 648518346420083590, 648518346420096646, 648518346420106118, 648518346420114054, 648518346420117382, 648518346420121990, 648518346420126086, 648518346420132998, 648518346420148614, 648518346420159878, 648518346420165766, 648518346420182150, 648518346420182406, 648518346420185478, 648518346420217222, 648518346420218502, 648518346420231558, 648518346420241286, 648518346420242822, 648518346420248198, 648518346420254342, 648518346420254598, 648518346420267654, 648518346420523654, 648518346420930360, 648518346421408288, 648518346421410336, 648518346422690336, 648518346422707232, 648518346422720544, 648518346422791712, 648518346422793504, 648518346422797600, 648518346422815264, 648518346422822944, 648518346422831392, 648518346422837280, 648518346422838816, 648518346422856736, 648518346422861088, 648518346422874400, 648518346422874656, 648518346422878240, 648518346422891296, 648518346422900256, 648518346422910240, 648518346422911520, 648518346422915398, 648518346422928672, 648518346422930464, 648518346422932294, 648518346422941216, 648518346422947872, 648518346422952736, 648518346423012128, 648518346423153732, 648518346423169604, 648518346423591530, 648518346424580286, 648518346425007743, 648518346425009279, 648518346425550220, 648518346425550732, 648518346425810981, 648518346425829413, 648518346425836581, 648518346425932407, 648518346425954423, 648518346425958519, 648518346425967735, 648518346425979511, 648518346425979767, 648518346425982071, 648518346425982583, 648518346425983351, 648518346425983607, 648518346426585604, 648518346426585860, 648518346426586372, 648518346426589444, 648518346426601732, 648518346426631940, 648518346426632196, 648518346426633476, 648518346426633732, 648518346426633988, 648518346426643204, 648518346426643460, 648518346426643716, 648518346426646788, 648518346426655236, 648518346426655492, 648518346426656516, 648518346426656772, 648518346426658052, 648518346426663428, 648518346427062797, 648518346427117581, 648518346427128589, 648518346427627358, 648518346427679873, 648518346427679959, 648518346427680897, 648518346428610047, 648518346428982939, 648518346428983195, 648518346429425819, 648518346429496447, 648518346429536639, 648518346429539199, 648518346429542271, 648518346429542783, 648518346429543039, 648518346429550463, 648518346429560959, 648518346429562155, 648518346429632555, 648518346429642027, 648518346429655595, 648518346429717968, 648518346429950938, 648518346429994283, 648518346429994795, 648518346429999403, 648518346430012203, 648518346430248389, 648518346430264005, 648518346430452080, 648518346430487407, 648518346430494063, 648518346430495599, 648518346430855397, 648518346430858213, 648518346430858469, 648518346431135856, 648518346431137648, 648518346431138160, 648518346431138416, 648518346431146864, 648518346431152496, 648518346431154032, 648518346431159664, 648518346431188848, 648518346431189104, 648518346431190128, 648518346431195248, 648518346431197296, 648518346431197552, 648518346431197808, 648518346431198064, 648518346431198832, 648518346431199088, 648518346431463599, 648518346431520511, 648518346431760638, 648518346431764734, 648518346431765502, 648518346431766270, 648518346431766526, 648518346431768062, 648518346431768318, 648518346431768574, 648518346431769598, 648518346432318230, 648518346432320790, 648518346432321302, 648518346432321814, 648518346432322326, 648518346432323350, 648518346432324118, 648518346432325654, 648518346432334102, 648518346432338198, 648518346432338454, 648518346432338710, 648518346432338966, 648518346432339478, 648518346432339734, 648518346432339990, 648518346432340246, 648518346432350742, 648518346432350998, 648518346432356374, 648518346432361238, 648518346432382486, 648518346432544999, 648518346432799744, 648518346432845824, 648518346433196896, 648518346433210720, 648518346433212256, 648518346433228312, 648518346433233432, 648518346433307928, 648518346433309464, 648518346433310232, 648518346434131224, 648518346434131480, 648518346434131736, 648518346434190360, 648518346434190872, 648518346434384968, 648518346434392798, 648518346434431560, 648518346434432072, 648518346434434376, 648518346434434632, 648518346434434888, 648518346434435144, 648518346434435656, 648518346434436680, 648518346434441032, 648518346434811825, 648518346434829489, 648518346435002415, 648518346435003183, 648518346435003951, 648518346435021359, 648518346435055583, 648518346435116751, 648518346435121615, 648518346435149519, 648518346435242975, 648518346436723188, 648518346436729076, 648518346436943031, 648518346436979127, 648518346437065376, 648518346437065888, 648518346437066656, 648518346437078432, 648518346437078688, 648518346437078944, 648518346437079200, 648518346437079968, 648518346437195508, 648518346437196020, 648518346437312865, 648518346437319605, 648518346437321141, 648518346437321653, 648518346437323361, 648518346437329077, 648518346437330357, 648518346437331381, 648518346437341537, 648518346437413806, 648518346437414318, 648518346437415342, 648518346437415598, 648518346437846731, 648518346437877451, 648518346437976725, 648518346438291946, 648518346438303210, 648518346438304234, 648518346438304490, 648518346438329636, 648518346438442532, 648518346438636836, 648518346438671563, 648518346438682059, 648518346438687012, 648518346438719780, 648518346438731812, 648518346438796836, 648518346438873544, 648518346438941005, 648518346438993997, 648518346439067213, 648518346439280840, 648518346439536843, 648518346439541451, 648518346439545547, 648518346439545803, 648518346439551179, 648518346439567563, 648518346439567819, 648518346439568075, 648518346439568331, 648518346439577035, 648518346439738869, 648518346439743477, 648518346439743989, 648518346439745525, 648518346439747317, 648518346439751157, 648518346439751669, 648518346439752437, 648518346439753717, 648518346439756789, 648518346439757045, 648518346439757301, 648518346439762165, 648518346439766005, 648518346439766517, 648518346439769589, 648518346439769845, 648518346439772149, 648518346439773173, 648518346439774197, 648518346439777269, 648518346439811829, 648518346439814389, 648518346439817717, 648518346439817973, 648518346439826933, 648518346439827189, 648518346439830261, 648518346439830517, 648518346439830773, 648518346439836405, 648518346439836661, 648518346439860469, 648518346439861749, 648518346440076533, 648518346440138485, 648518346440147445, 648518346440802179, 648518346440812675, 648518346440831363, 648518346440832387, 648518346440832899, 648518346440834179, 648518346440836843, 648518346440851075, 648518346440851331, 648518346440851587, 648518346440851843, 648518346440852355, 648518346440853891, 648518346440854403, 648518346440854915, 648518346440856963, 648518346440862339, 648518346440863107, 648518346440885123, 648518346440885635, 648518346440885891, 648518346440886659, 648518346440886915, 648518346440887171, 648518346440930947, 648518346440931459, 648518346440931715, 648518346440931971, 648518346440934379, 648518346440936683, 648518346440938731, 648518346440940011, 648518346440966123, 648518346441675179, 648518346441677995, 648518346441716651, 648518346441718443, 648518346441742059, 648518346441742315, 648518346441742571, 648518346441742827, 648518346441744107, 648518346441745131, 648518346441746923, 648518346441754091, 648518346441754603, 648518346441754859, 648518346441755563, 648518346441756075, 648518346441756331, 648518346441757099, 648518346441757611, 648518346441770731, 648518346441812459, 648518346441813227, 648518346441813995, 648518346441822699, 648518346441824235, 648518346441824491, 648518346441825003, 648518346441825259, 648518346441825515, 648518346441826027, 648518346441826283, 648518346441826539, 648518346441826795, 648518346442201087, 648518346442202879, 648518346442215935, 648518346442216447, 648518346442485247, 648518346442486015, 648518346443976913, 648518346443980497, 648518346443981265, 648518346444261841, 648518346444307921, 648518346444391105, 648518346444393153, 648518346444393409, 648518346444394177, 648518346444395713, 648518346444397505, 648518346444398785, 648518346444400321, 648518346444417729, 648518346444419265, 648518346444431809, 648518346444439745, 648518346444443585, 648518346444446401, 648518346444446657, 648518346444446929, 648518346444466641, 648518346444745681, 648518346444755409, 648518346444970812, 648518346445214012, 648518346445224508, 648518346445224764, 648518346445594642, 648518346445595154, 648518346445635346, 648518346445739580, 648518346445739836, 648518346445740860, 648518346445741116, 648518346445741372, 648518346445741628, 648518346446218812, 648518346446219068, 648518346446219580, 648518346446220604, 648518346446226236, 648518346446228284, 648518346446228540, 648518346446229052, 648518346447062009, 648518346447063289, 648518346447064313, 648518346447139577, 648518346447139833, 648518346447142649, 648518346447142905, 648518346447143929, 648518346447164665, 648518346447165945, 648518346447182841, 648518346447378169, 648518346447687417, 648518346447687673, 648518346447688441, 648518346447689465, 648518346447692537, 648518346447695097, 648518346447695609, 648518346447696633, 648518346447699961, 648518346447700473, 648518346447944953]
# [648518346455281550, 648518346437816989, 648518346441496546]
# [648518346441496546]
# [648518346437816989]
# [648518346433128630]
# [648518346438847348, 648518346440196004, 648518346442083591, 648518346444263624, 648518346444502503, 648518346444541513, 648518346447553683, 648518346450474414, 648518346451148019, 648518346452120559]
# [648518346345876117, 648518346345877397, 648518346345889685, 648518346345896597, 648518346345937557, 648518346346200725, 648518346346625496, 648518346346652888, 648518346346654168, 648518346346654424, 648518346346655704, 648518346346676696, 648518346346686424, 648518346346687704, 648518346347537921, 648518346348288312, 648518346349014266, 648518346349055738, 648518346363987489, 648518346366326409, 648518346367030414, 648518346367553074, 648518346367569270, 648518346367882034, 648518346368752592, 648518346368752848, 648518346368769744, 648518346368771536, 648518346368811216, 648518346368816336, 648518346368819408, 648518346368884233, 648518346402365869, 648518346415490511, 648518346423846943, 648518346423849247, 648518346430865094, 648518346434105735, 648518346434159751, 648518346435209412, 648518346435211972, 648518346435215044, 648518346435268548, 648518346435336388, 648518346435337668, 648518346435561838, 648518346435565678, 648518346435569518, 648518346435886958, 648518346436615534, 648518346436880135, 648518346436931950, 648518346436970862, 648518346437714222, 648518346438403901, 648518346438568557, 648518346440301897, 648518346440421449, 648518346440433225, 648518346440757833, 648518346440762697, 648518346440764233, 648518346440771657, 648518346440773449, 648518346440850249, 648518346440943945, 648518346442063185, 648518346442645807, 648518346443339976, 648518346443646920, 648518346443653832, 648518346443664840, 648518346443803336, 648518346443803592, 648518346443804104, 648518346443823816, 648518346443831752, 648518346444717749, 648518346447553683, 648518346474305190]
# [648518346432942636, 648518346433131820, 648518346434827178, 648518346434833697, 648518346435711615, 648518346436710014, 648518346436786455, 648518346436886493, 648518346437275258, 648518346437391963, 648518346437491322, 648518346437789085, 648518346438002007, 648518346438018391, 648518346438207912, 648518346439531100, 648518346440149399, 648518346440196004, 648518346440661519, 648518346440942118, 648518346441331671, 648518346441578715, 648518346442290480, 648518346443454157, 648518346443853177, 648518346444314087, 648518346444521575, 648518346444531440, 648518346445043229, 648518346446171360, 648518346446630441, 648518346447553683, 648518346449080943, 648518346451295035, 648518346451904761, 648518346452485350, 648518346454034369, 648518346462161539, 648518346462259331, 648518346469538093] # Top postsyn neurons to 648518346444314087 that have a nucleus in the volume
# [648518346429302836, 648518346432510775, 648518346435068156, 648518346436621392, 648518346436629840, 648518346436698777, 648518346436797822, 648518346436805246, 648518346438828916, 648518346439575644, 648518346440739174, 648518346441158615, 648518346441603042, 648518346441998053, 648518346442010597, 648518346443207945, 648518346444264392, 648518346444314087, 648518346445010973, 648518346446683356, 648518346446690524, 648518346447910219, 648518346448038906, 648518346449469764, 648518346454927753, 648518346478162598] # 26 pyr neurons with large nuc vol from annotated pyr set
# root_ids = [648518346432510775, 648518346436805246, 648518346440739174, 648518346441158615, 648518346441998053, 648518346444264392, 648518346445010973, 648518346448038906, 648518346449469764] # set of pyr neurons that span the ca3 area
# [648518346444314087] # CA3 pyr neuron with an axon in the volume 
# [648518346424844545, 648518346433641043, 648518346433752659, 648518346437104007, 648518346438222248, 648518346438737689, 648518346439852695, 648518346439990677, 648518346440208816, 648518346441591259, 648518346443969657, 648518346445848425, 648518346445955534, 648518346451989245, 648518346455348332] # Top 15 non-annotated root ids by nuclei vol sum size; originally top 20 but removed 1 vasc and 4 artifactual segments
# [648518346431533987, 648518346431683520, 648518346436695831, 648518346437232646, 648518346437496155, 648518346439575644, 648518346440734822, 648518346440738406, 648518346440894353, 648518346440942118, 648518346442010597, 648518346442419701, 648518346442571721, 648518346444557287, 648518346447704211, 648518346451022260, 648518346451361634, 648518346452485350, 648518346454678665, 648518346465796078] # Top 20 annotated neurons by nuclei volume sum
#[648518346478068134]
#[648518346447140960, 648518346441839621, 648518346447553683]
#[648518346342665139, 648518346342702003, 648518346342732979, 648518346342744755, 648518346343511365, 648518346343533637, 648518346343616069, 648518346343889862, 648518346344623525, 648518346344711589, 648518346344728997, 648518346344729253, 648518346345505999, 648518346347350822, 648518346349439002, 648518346349520922, 648518346349530129, 648518346349542682, 648518346349560346, 648518346349570074, 648518346349572122, 648518346349579546, 648518346349597978, 648518346349598490, 648518346349689105, 648518346349737233, 648518346349746449, 648518346349810449, 648518346349847313, 648518346349848081, 648518346349860113, 648518346349912593, 648518346350002513, 648518346350081690, 648518346350702402, 648518346350711874, 648518346350712898, 648518346350737474, 648518346350763586, 648518346359532259, 648518346361323537, 648518346364252949, 648518346364947081, 648518346364951945, 648518346364954761, 648518346364959881, 648518346364961161, 648518346365420851, 648518346365750097, 648518346367660462, 648518346367663790, 648518346367850482, 648518346367869170, 648518346367925746, 648518346367927026, 648518346368533630, 648518346368807038, 648518346368808574, 648518346368809086, 648518346368810878, 648518346368828542, 648518346368830590, 648518346368831614, 648518346368859006, 648518346368886616, 648518346369007448, 648518346369023576, 648518346369199132, 648518346369209628, 648518346369214748, 648518346369219868, 648518346369225756, 648518346369346588, 648518346369741076, 648518346371404188, 648518346375733093, 648518346375733349, 648518346379694032, 648518346379764944, 648518346379789776, 648518346380112551, 648518346380160423, 648518346380666637, 648518346380695565, 648518346380745485, 648518346381678863, 648518346381756943, 648518346381760527, 648518346381779983, 648518346381786992, 648518346381814799, 648518346381821808, 648518346381842959, 648518346381925752, 648518346381927288, 648518346383516190, 648518346383685665, 648518346383998334, 648518346384049165, 648518346385051941, 648518346385107562, 648518346385173610, 648518346385590321, 648518346386387139, 648518346387098151, 648518346387152935, 648518346387391012, 648518346387679727, 648518346387806247, 648518346387838703, 648518346388596315, 648518346388609115, 648518346388616539, 648518346388679259, 648518346389098795, 648518346389172888, 648518346389248423, 648518346389254066, 648518346389255346, 648518346389260967, 648518346389978597, 648518346389981413, 648518346390379749, 648518346391087684, 648518346392838520, 648518346393911282, 648518346393914359, 648518346394245346, 648518346394263435, 648518346394273419, 648518346394302091, 648518346394432327, 648518346394445639, 648518346394446151, 648518346394476939, 648518346394513547, 648518346394516875, 648518346394536331, 648518346394540683, 648518346394545035, 648518346395044109, 648518346395103458, 648518346395153933, 648518346395169250, 648518346395185933, 648518346395188749, 648518346395194381, 648518346395196685, 648518346395199757, 648518346395202530, 648518346395239693, 648518346395249421, 648518346396410995, 648518346396744052, 648518346397489978, 648518346397713711, 648518346397909551, 648518346397910831, 648518346398073871, 648518346398098191, 648518346398120975, 648518346398402607, 648518346401295964, 648518346401761033, 648518346401784841, 648518346401791753, 648518346401836809, 648518346401866249, 648518346402036745, 648518346402038537, 648518346402043145, 648518346402069001, 648518346402073097, 648518346402089737, 648518346402091529, 648518346402093577, 648518346402099465, 648518346402121993, 648518346402125577, 648518346402126601, 648518346402129417, 648518346402156041, 648518346402173961, 648518346402184713, 648518346402190345, 648518346402236169, 648518346402876996, 648518346403697068, 648518346403719852, 648518346403789740, 648518346403813036, 648518346403898284, 648518346403920812, 648518346403927980, 648518346403931820, 648518346403935404, 648518346403940524, 648518346403987116, 648518346403988140, 648518346403995308, 648518346407959316, 648518346407959828, 648518346407962132, 648518346407971860, 648518346407972372, 648518346408049428, 648518346408105492, 648518346408251924, 648518346408253716, 648518346408588052, 648518346408622356, 648518346408681236, 648518346408762391, 648518346408800023, 648518346408912663, 648518346408916247, 648518346410987829, 648518346411284021, 648518346411286581, 648518346411291701, 648518346411301941, 648518346411309365, 648518346411322421, 648518346411326005, 648518346411328053, 648518346411333941, 648518346411335989, 648518346411338549, 648518346411511990, 648518346411512246, 648518346411531446, 648518346411532982, 648518346411536566, 648518346411537590, 648518346411664566, 648518346411753243, 648518346411756315, 648518346411774235, 648518346411796763, 648518346411798299, 648518346411798811, 648518346411804443, 648518346411815195, 648518346413234542, 648518346414303903, 648518346414312095, 648518346414328479, 648518346414330271, 648518346414336415, 648518346414340511, 648518346414343327, 648518346414347935, 648518346414538185, 648518346414584993, 648518346415254002, 648518346415375762, 648518346415412114, 648518346415448978, 648518346415492754, 648518346415496946, 648518346415521946, 648518346415522714, 648518346415560346, 648518346415582618, 648518346415595674, 648518346415597210, 648518346415600794, 648518346415601562, 648518346415602074, 648518346415606938, 648518346415613082, 648518346415613850, 648518346415819762, 648518346415828466, 648518346415832562, 648518346416064154, 648518346416285637, 648518346417379078, 648518346417381382, 648518346417382918, 648518346417389830, 648518346417392134, 648518346417392390, 648518346417443078, 648518346417447174, 648518346417467398, 648518346417468934, 648518346417470470, 648518346417470726, 648518346417471238, 648518346417476870, 648518346417477894, 648518346417479174, 648518346417481734, 648518346417496326, 648518346417761798, 648518346417931363, 648518346418703513, 648518346418927867, 648518346418948603, 648518346418989495, 648518346418993915, 648518346419069179, 648518346419080699, 648518346419084539, 648518346419090683, 648518346419092475, 648518346419102715, 648518346419113979, 648518346419114491, 648518346419141627, 648518346419966398, 648518346419999678, 648518346420462270, 648518346421175214, 648518346421290926, 648518346421400863, 648518346421965042, 648518346421992032, 648518346423507030, 648518346424221596, 648518346425398476, 648518346425550709, 648518346425551477, 648518346425845293, 648518346426062760, 648518346426073256, 648518346426074536, 648518346426074792, 648518346426240557, 648518346426244653, 648518346426548781, 648518346426995984, 648518346427053509, 648518346427078445, 648518346427092933, 648518346427436319, 648518346427560223, 648518346427761288, 648518346428908800, 648518346429422818, 648518346429423074, 648518346429447650, 648518346429804373, 648518346430233918, 648518346431070405, 648518346431467641, 648518346431702143, 648518346432602270, 648518346433778919, 648518346436789894, 648518346437702179, 648518346437702634, 648518346437775771, 648518346437838570, 648518346437841898, 648518346437866474, 648518346437870314, 648518346438668086, 648518346447083078, 648518346447141507, 648518346447330630, 648518346447398470]
# [648518346341734247, 648518346342407091, 648518346342407859, 648518346342409907, 648518346342634279, 648518346342665139, 648518346342672051, 648518346342676147, 648518346342682547, 648518346342684595, 648518346342687155, 648518346342689971, 648518346342702003, 648518346342732979, 648518346342743987, 648518346342744755, 648518346342771379, 648518346342782899, 648518346342956926, 648518346343201861, 648518346343364677, 648518346343511365, 648518346343533637, 648518346343613509, 648518346343616069, 648518346343620677, 648518346343889862, 648518346344326397, 648518346344621733, 648518346344623525, 648518346344647077, 648518346344648101, 648518346344682149, 648518346344683941, 648518346344689061, 648518346344709541, 648518346344711589, 648518346344713125, 648518346344728997, 648518346344729253, 648518346345505999, 648518346345813389, 648518346346769314, 648518346346905624, 648518346347327782, 648518346347335206, 648518346347350822, 648518346347461366, 648518346348520683, 648518346349431322, 648518346349439002, 648518346349448218, 648518346349453594, 648518346349468186, 648518346349473050, 648518346349476890, 648518346349484314, 648518346349490714, 648518346349520922, 648518346349530129, 648518346349535514, 648518346349540378, 648518346349542682, 648518346349546266, 648518346349548826, 648518346349551898, 648518346349555738, 648518346349555994, 648518346349557274, 648518346349560346, 648518346349561114, 648518346349562138, 648518346349566234, 648518346349567770, 648518346349570074, 648518346349572122, 648518346349579546, 648518346349584922, 648518346349586970, 648518346349591578, 648518346349595418, 648518346349597978, 648518346349598490, 648518346349601562, 648518346349605658, 648518346349615642, 648518346349616154, 648518346349689105, 648518346349737233, 648518346349746449, 648518346349754385, 648518346349806097, 648518346349807121, 648518346349810193, 648518346349810449, 648518346349811985, 648518346349815313, 648518346349822225, 648518346349822481, 648518346349826065, 648518346349827345, 648518346349831953, 648518346349832465, 648518346349834001, 648518346349844497, 648518346349847313, 648518346349848081, 648518346349855249, 648518346349860113, 648518346349888017, 648518346349895697, 648518346349912090, 648518346349912593, 648518346349928986, 648518346350002513, 648518346350003025, 648518346350081690, 648518346350105498, 648518346350115738, 648518346350626626, 648518346350645826, 648518346350649410, 648518346350654018, 648518346350660674, 648518346350663234, 648518346350667842, 648518346350682690, 648518346350702402, 648518346350708034, 648518346350710850, 648518346350711874, 648518346350712898, 648518346350713922, 648518346350714690, 648518346350714946, 648518346350716482, 648518346350718786, 648518346350728770, 648518346350737474, 648518346350760514, 648518346350761538, 648518346350763586, 648518346350764866, 648518346350771010, 648518346350777666, 648518346350778178, 648518346350781762, 648518346350801218, 648518346350802242, 648518346350803010, 648518346359448035, 648518346359532259, 648518346361323537, 648518346361876303, 648518346362245894, 648518346364065090, 648518346364176727, 648518346364252949, 648518346364775561, 648518346364776841, 648518346364798857, 648518346364799113, 648518346364800905, 648518346364801161, 648518346364946313, 648518346364947081, 648518346364947337, 648518346364947593, 648518346364948105, 648518346364948873, 648518346364950153, 648518346364950409, 648518346364950665, 648518346364951945, 648518346364952457, 648518346364953993, 648518346364954761, 648518346364955273, 648518346364958857, 648518346364959113, 648518346364959369, 648518346364959625, 648518346364959881, 648518346364960137, 648518346364960393, 648518346364960649, 648518346364961161, 648518346364961417, 648518346364961673, 648518346364965257, 648518346364965513, 648518346364966537, 648518346364967305, 648518346364967561, 648518346364969865, 648518346364970633, 648518346364971913, 648518346364972681, 648518346365420851, 648518346365426227, 648518346365542793, 648518346365546547, 648518346365547571, 648518346365586739, 648518346365589299, 648518346365589811, 648518346365590579, 648518346365593139, 648518346365750097, 648518346365991987, 648518346367136231, 648518346367336690, 648518346367624110, 648518346367644334, 648518346367644846, 648518346367658158, 648518346367658926, 648518346367659694, 648518346367660206, 648518346367660462, 648518346367663790, 648518346367849202, 648518346367850482, 648518346367868914, 648518346367869170, 648518346367870962, 648518346367871986, 648518346367873778, 648518346367879666, 648518346367925746, 648518346367927026, 648518346368532350, 648518346368533118, 648518346368533630, 648518346368547454, 648518346368547966, 648518346368806270, 648518346368807038, 648518346368807550, 648518346368808062, 648518346368808574, 648518346368808830, 648518346368809086, 648518346368810622, 648518346368810878, 648518346368811390, 648518346368811646, 648518346368813182, 648518346368814718, 648518346368815230, 648518346368824958, 648518346368825470, 648518346368826750, 648518346368827006, 648518346368828542, 648518346368828798, 648518346368830590, 648518346368831614, 648518346368833918, 648518346368834430, 648518346368834942, 648518346368859006, 648518346368861054, 648518346368866430, 648518346368886616, 648518346368887384, 648518346368997720, 648518346369007448, 648518346369023576, 648518346369166620, 648518346369167388, 648518346369168156, 648518346369168412, 648518346369169436, 648518346369169948, 648518346369170972, 648518346369171228, 648518346369171484, 648518346369171740, 648518346369186076, 648518346369187612, 648518346369188124, 648518346369194780, 648518346369196828, 648518346369197084, 648518346369199132, 648518346369204252, 648518346369206300, 648518346369206556, 648518346369206812, 648518346369208348, 648518346369209628, 648518346369209884, 648518346369211420, 648518346369212188, 648518346369212700, 648518346369213212, 648518346369213724, 648518346369214748, 648518346369216284, 648518346369217052, 648518346369218588, 648518346369218844, 648518346369219868, 648518346369224476, 648518346369225756, 648518346369253404, 648518346369288770, 648518346369346588, 648518346369741076, 648518346369929816, 648518346370663658, 648518346370712637, 648518346371137304, 648518346371404188, 648518346371637942, 648518346372352808, 648518346372444097, 648518346374992023, 648518346375134871, 648518346375227031, 648518346375648101, 648518346375733093, 648518346375733349, 648518346375735397, 648518346376411557, 648518346377187407, 648518346377446431, 648518346379680063, 648518346379690960, 648518346379694032, 648518346379696336, 648518346379705552, 648518346379733200, 648518346379739856, 648518346379747536, 648518346379764944, 648518346379789776, 648518346379808720, 648518346379884032, 648518346380011776, 648518346380112551, 648518346380160423, 648518346380666637, 648518346380692493, 648518346380693773, 648518346380695565, 648518346380745229, 648518346380745485, 648518346380756749, 648518346380786189, 648518346380821517, 648518346380827661, 648518346381053527, 648518346381489890, 648518346381645707, 648518346381674255, 648518346381678863, 648518346381679119, 648518346381704975, 648518346381744655, 648518346381748495, 648518346381753359, 648518346381756943, 648518346381760527, 648518346381779312, 648518346381779983, 648518346381786992, 648518346381814799, 648518346381821808, 648518346381825648, 648518346381825807, 648518346381829903, 648518346381839728, 648518346381841167, 648518346381842959, 648518346381897336, 648518346381911928, 648518346381920632, 648518346381925752, 648518346381927288, 648518346381982584, 648518346382065455, 648518346382538849, 648518346383277451, 648518346383280011, 648518346383280267, 648518346383516190, 648518346383517470, 648518346383626490, 648518346383685665, 648518346383711265, 648518346383718945, 648518346383724577, 648518346383726625, 648518346383745569, 648518346383792417, 648518346383793671, 648518346383797374, 648518346383978878, 648518346383998334, 648518346384010366, 648518346384025214, 648518346384049165, 648518346384220941, 648518346384896412, 648518346385051941, 648518346385107562, 648518346385148522, 648518346385149034, 648518346385173610, 648518346385590321, 648518346386194512, 648518346386339595, 648518346386375691, 648518346386380555, 648518346386382091, 648518346386387139, 648518346386779403, 648518346387098151, 648518346387152935, 648518346387238436, 648518346387383588, 648518346387391012, 648518346387679727, 648518346387700975, 648518346387726876, 648518346387806247, 648518346387807015, 648518346387809575, 648518346387838703, 648518346387952738, 648518346388100424, 648518346388218753, 648518346388589403, 648518346388591707, 648518346388591963, 648518346388592219, 648518346388592475, 648518346388596315, 648518346388609115, 648518346388611931, 648518346388616027, 648518346388616539, 648518346388619099, 648518346388623195, 648518346388628315, 648518346388642271, 648518346388648027, 648518346388664167, 648518346388664679, 648518346388679259, 648518346389002663, 648518346389003431, 648518346389014951, 648518346389047463, 648518346389098795, 648518346389136319, 648518346389172888, 648518346389192871, 648518346389197208, 648518346389203096, 648518346389220775, 648518346389225895, 648518346389227431, 648518346389229735, 648518346389235367, 648518346389237671, 648518346389240487, 648518346389248423, 648518346389253554, 648518346389254066, 648518346389255346, 648518346389260967, 648518346389915663, 648518346389925647, 648518346389931279, 648518346389941263, 648518346389941775, 648518346389942031, 648518346389944591, 648518346389976335, 648518346389978597, 648518346389981413, 648518346389998565, 648518346390379749, 648518346390834244, 648518346391087684, 648518346391092292, 648518346391283509, 648518346391320388, 648518346391853131, 648518346392832632, 648518346392838520, 648518346393790450, 648518346393795826, 648518346393838066, 648518346393910258, 648518346393911282, 648518346393914359, 648518346393914871, 648518346393915127, 648518346393948146, 648518346393954034, 648518346393967402, 648518346393969394, 648518346393973746, 648518346394002474, 648518346394242760, 648518346394245346, 648518346394255243, 648518346394260619, 648518346394263435, 648518346394273419, 648518346394302091, 648518346394427975, 648518346394428999, 648518346394429511, 648518346394432327, 648518346394445639, 648518346394446151, 648518346394476939, 648518346394495303, 648518346394504843, 648518346394513547, 648518346394516107, 648518346394516875, 648518346394536331, 648518346394540683, 648518346394545035, 648518346395036386, 648518346395036642, 648518346395044109, 648518346395051277, 648518346395054349, 648518346395056610, 648518346395060237, 648518346395078370, 648518346395086861, 648518346395103458, 648518346395140365, 648518346395153933, 648518346395160333, 648518346395163917, 648518346395164429, 648518346395165965, 648518346395167202, 648518346395169250, 648518346395181325, 648518346395184653, 648518346395185933, 648518346395187981, 648518346395188749, 648518346395189773, 648518346395193613, 648518346395194381, 648518346395196685, 648518346395199757, 648518346395201805, 648518346395202530, 648518346395204109, 648518346395205133, 648518346395205901, 648518346395209485, 648518346395211789, 648518346395212557, 648518346395219725, 648518346395229197, 648518346395229709, 648518346395233293, 648518346395234061, 648518346395239181, 648518346395239693, 648518346395240205, 648518346395242509, 648518346395244557, 648518346395249421, 648518346395251213, 648518346396410995, 648518346396660832, 648518346396744052, 648518346397489978, 648518346397710639, 648518346397710895, 648518346397713711, 648518346397909551, 648518346397909807, 648518346397910831, 648518346398004271, 648518346398073615, 648518346398073871, 648518346398084399, 648518346398098191, 648518346398103055, 648518346398120975, 648518346398124303, 648518346398124559, 648518346398402607, 648518346398408239, 648518346400232625, 648518346400234417, 648518346401295964, 648518346401761033, 648518346401765385, 648518346401773065, 648518346401784841, 648518346401791753, 648518346401806601, 648518346401836809, 648518346401866249, 648518346401867529, 648518346402036745, 648518346402038025, 648518346402038537, 648518346402043145, 648518346402069001, 648518346402071561, 648518346402073097, 648518346402076681, 648518346402084873, 648518346402088457, 648518346402088713, 648518346402089737, 648518346402091529, 648518346402093577, 648518346402099465, 648518346402110985, 648518346402113033, 648518346402116105, 648518346402118409, 648518346402120457, 648518346402121993, 648518346402125577, 648518346402126601, 648518346402129417, 648518346402151433, 648518346402156041, 648518346402157065, 648518346402158345, 648518346402173961, 648518346402184713, 648518346402186249, 648518346402190345, 648518346402197513, 648518346402204681, 648518346402219529, 648518346402236169, 648518346402238729, 648518346402502212, 648518346402778948, 648518346402876996, 648518346403608047, 648518346403677356, 648518346403695276, 648518346403697068, 648518346403719852, 648518346403757996, 648518346403758252, 648518346403759020, 648518346403789740, 648518346403813036, 648518346403832236, 648518346403834028, 648518346403834796, 648518346403897516, 648518346403898284, 648518346403900076, 648518346403900588, 648518346403904684, 648518346403911340, 648518346403920812, 648518346403925164, 648518346403927980, 648518346403929004, 648518346403931820, 648518346403934636, 648518346403935404, 648518346403935916, 648518346403938732, 648518346403940524, 648518346403949996, 648518346403950252, 648518346403957164, 648518346403973292, 648518346403973548, 648518346403986348, 648518346403987116, 648518346403988140, 648518346403992748, 648518346403995308, 648518346403995820, 648518346404039340, 648518346404055980, 648518346404058796, 648518346404078252, 648518346404340199, 648518346407959316, 648518346407959828, 648518346407962132, 648518346407971860, 648518346407972372, 648518346407986452, 648518346407989524, 648518346407989780, 648518346407991828, 648518346407992852, 648518346408049428, 648518346408064276, 648518346408105492, 648518346408251668, 648518346408251924, 648518346408253716, 648518346408254996, 648518346408588052, 648518346408621332, 648518346408622356, 648518346408681236, 648518346408735767, 648518346408740631, 648518346408749335, 648518346408749591, 648518346408762391, 648518346408800023, 648518346408891159, 648518346408892951, 648518346408912663, 648518346408916247, 648518346410984501, 648518346410986037, 648518346410986549, 648518346410987317, 648518346410987829, 648518346410989621, 648518346411043283, 648518346411165015, 648518346411276085, 648518346411280693, 648518346411284021, 648518346411286325, 648518346411286581, 648518346411290421, 648518346411291701, 648518346411292213, 648518346411293493, 648518346411294261, 648518346411294773, 648518346411298869, 648518346411300149, 648518346411300405, 648518346411300917, 648518346411301941, 648518346411303221, 648518346411309365, 648518346411310645, 648518346411314485, 648518346411314741, 648518346411321909, 648518346411322421, 648518346411325749, 648518346411326005, 648518346411326773, 648518346411328053, 648518346411333941, 648518346411335989, 648518346411338549, 648518346411344693, 648518346411350069, 648518346411511990, 648518346411512246, 648518346411513014, 648518346411518646, 648518346411519414, 648518346411528886, 648518346411530422, 648518346411531446, 648518346411532214, 648518346411532726, 648518346411532982, 648518346411533750, 648518346411534006, 648518346411536566, 648518346411537078, 648518346411537590, 648518346411564726, 648518346411571382, 648518346411574710, 648518346411575990, 648518346411576758, 648518346411577014, 648518346411578294, 648518346411579574, 648518346411582134, 648518346411584182, 648518346411588278, 648518346411589558, 648518346411664566, 648518346411721398, 648518346411723190, 648518346411744027, 648518346411747611, 648518346411748379, 648518346411749403, 648518346411753243, 648518346411754011, 648518346411755803, 648518346411756315, 648518346411765275, 648518346411766555, 648518346411767067, 648518346411770139, 648518346411770395, 648518346411774235, 648518346411774747, 648518346411775003, 648518346411775259, 648518346411775515, 648518346411777051, 648518346411790875, 648518346411793179, 648518346411793947, 648518346411794203, 648518346411796763, 648518346411798299, 648518346411798811, 648518346411799579, 648518346411799835, 648518346411800603, 648518346411804443, 648518346411809307, 648518346411812379, 648518346411813915, 648518346411815195, 648518346413234542, 648518346413279598, 648518346413291886, 648518346413297774, 648518346413301358, 648518346413327470, 648518346413335662, 648518346413353838, 648518346413389422, 648518346414301343, 648518346414302623, 648518346414303903, 648518346414312095, 648518346414321567, 648518346414322591, 648518346414324383, 648518346414324639, 648518346414324895, 648518346414328479, 648518346414328991, 648518346414330015, 648518346414330271, 648518346414334367, 648518346414335647, 648518346414336415, 648518346414340511, 648518346414340767, 648518346414341023, 648518346414343327, 648518346414347935, 648518346414354591, 648518346414357919, 648518346414358175, 648518346414359711, 648518346414360223, 648518346414380703, 648518346414382495, 648518346414385567, 648518346414505533, 648518346414538185, 648518346414584993, 648518346414908094, 648518346414978198, 648518346415221234, 648518346415222258, 648518346415253746, 648518346415254002, 648518346415272434, 648518346415280114, 648518346415282162, 648518346415288818, 648518346415321744, 648518346415375762, 648518346415410578, 648518346415412114, 648518346415414930, 648518346415421072, 648518346415445906, 648518346415446162, 648518346415446930, 648518346415448978, 648518346415470226, 648518346415489682, 648518346415492754, 648518346415495154, 648518346415496946, 648518346415507602, 648518346415521434, 648518346415521690, 648518346415521946, 648518346415522714, 648518346415522970, 648518346415523226, 648518346415523482, 648518346415524506, 648518346415524762, 648518346415531397, 648518346415560346, 648518346415561882, 648518346415562906, 648518346415563162, 648518346415563674, 648518346415563930, 648518346415565722, 648518346415567002, 648518346415581594, 648518346415582618, 648518346415583130, 648518346415583898, 648518346415584922, 648518346415591066, 648518346415591322, 648518346415593114, 648518346415593882, 648518346415594138, 648518346415595674, 648518346415597210, 648518346415599002, 648518346415599514, 648518346415599770, 648518346415600026, 648518346415600282, 648518346415600538, 648518346415600794, 648518346415601562, 648518346415602074, 648518346415603098, 648518346415603866, 648518346415605658, 648518346415605914, 648518346415606682, 648518346415606938, 648518346415607450, 648518346415611290, 648518346415612058, 648518346415613082, 648518346415613850, 648518346415614106, 648518346415702514, 648518346415703026, 648518346415703282, 648518346415704818, 648518346415706610, 648518346415707378, 648518346415818994, 648518346415819762, 648518346415826674, 648518346415828466, 648518346415832562, 648518346416045978, 648518346416064154, 648518346416285637, 648518346416348914, 648518346417379078, 648518346417380614, 648518346417380870, 648518346417381382, 648518346417382406, 648518346417382918, 648518346417383686, 648518346417384454, 648518346417384710, 648518346417384966, 648518346417385222, 648518346417385478, 648518346417385734, 648518346417385990, 648518346417386502, 648518346417389830, 648518346417390086, 648518346417391110, 648518346417391366, 648518346417392134, 648518346417392390, 648518346417438982, 648518346417443078, 648518346417445382, 648518346417447174, 648518346417447686, 648518346417449222, 648518346417465094, 648518346417465350, 648518346417465606, 648518346417467398, 648518346417468166, 648518346417468934, 648518346417469446, 648518346417469702, 648518346417469958, 648518346417470470, 648518346417470726, 648518346417471238, 648518346417475846, 648518346417476358, 648518346417476870, 648518346417477126, 648518346417477894, 648518346417478918, 648518346417479174, 648518346417480710, 648518346417481222, 648518346417481734, 648518346417483270, 648518346417483782, 648518346417485830, 648518346417487622, 648518346417487878, 648518346417496326, 648518346417496838, 648518346417761798, 648518346417931363, 648518346418019904, 648518346418195353, 648518346418205520, 648518346418264912, 648518346418498419, 648518346418560883, 648518346418703513, 648518346418912763, 648518346418913787, 648518346418914043, 648518346418914299, 648518346418926331, 648518346418926587, 648518346418927355, 648518346418927611, 648518346418927867, 648518346418928379, 648518346418948603, 648518346418989495, 648518346418991799, 648518346418993915, 648518346418994171, 648518346419001783, 648518346419058871, 648518346419068667, 648518346419069179, 648518346419070971, 648518346419074043, 648518346419076091, 648518346419077115, 648518346419080699, 648518346419082491, 648518346419083003, 648518346419083771, 648518346419084539, 648518346419086843, 648518346419089147, 648518346419090683, 648518346419091451, 648518346419092475, 648518346419092731, 648518346419092987, 648518346419096059, 648518346419099643, 648518346419099899, 648518346419100411, 648518346419101179, 648518346419101435, 648518346419102715, 648518346419103227, 648518346419103995, 648518346419112443, 648518346419112699, 648518346419112955, 648518346419113467, 648518346419113979, 648518346419114491, 648518346419116027, 648518346419119611, 648518346419141627, 648518346419342283, 648518346419513890, 648518346419949758, 648518346419950526, 648518346419951038, 648518346419966398, 648518346419966654, 648518346419967166, 648518346419967934, 648518346419994302, 648518346419999678, 648518346420009662, 648518346420009918, 648518346420021182, 648518346420161726, 648518346420162494, 648518346420385982, 648518346420386750, 648518346420395966, 648518346420396478, 648518346420396990, 648518346420416446, 648518346420424382, 648518346420446654, 648518346420462270, 648518346420972236, 648518346421175214, 648518346421189514, 648518346421214126, 648518346421214382, 648518346421260462, 648518346421261998, 648518346421263022, 648518346421290926, 648518346421371423, 648518346421371679, 648518346421372447, 648518346421372703, 648518346421380383, 648518346421381663, 648518346421381919, 648518346421383455, 648518346421383967, 648518346421384735, 648518346421400863, 648518346421401119, 648518346421965042, 648518346421992032, 648518346422027439, 648518346422028207, 648518346422028463, 648518346422033071, 648518346422033583, 648518346422049455, 648518346422067631, 648518346422068655, 648518346422098607, 648518346423499606, 648518346423500118, 648518346423500630, 648518346423500886, 648518346423501142, 648518346423501910, 648518346423502166, 648518346423502422, 648518346423502678, 648518346423502934, 648518346423503446, 648518346423504214, 648518346423504470, 648518346423505238, 648518346423505494, 648518346423506006, 648518346423507030, 648518346423507286, 648518346423507798, 648518346423508054, 648518346423508310, 648518346423508822, 648518346423509078, 648518346423511382, 648518346423511894, 648518346423512150, 648518346423512406, 648518346423513174, 648518346423515990, 648518346423543126, 648518346423595094, 648518346423595350, 648518346423600214, 648518346423642667, 648518346423792064, 648518346423824064, 648518346423824832, 648518346423846774, 648518346423860342, 648518346423880035, 648518346424066516, 648518346424215196, 648518346424215452, 648518346424220316, 648518346424221596, 648518346424223132, 648518346424442140, 648518346424916505, 648518346424986548, 648518346425040052, 648518346425056425, 648518346425056937, 648518346425057424, 648518346425114256, 648518346425398476, 648518346425398732, 648518346425464692, 648518346425533045, 648518346425533301, 648518346425550453, 648518346425550709, 648518346425550965, 648518346425551477, 648518346425588781, 648518346425716125, 648518346425742493, 648518346425845293, 648518346425846061, 648518346425847341, 648518346426044974, 648518346426054824, 648518346426056616, 648518346426058920, 648518346426060456, 648518346426060712, 648518346426061224, 648518346426062248, 648518346426062760, 648518346426073256, 648518346426073512, 648518346426074536, 648518346426074792, 648518346426075304, 648518346426075816, 648518346426076584, 648518346426078509, 648518346426102830, 648518346426144045, 648518346426144813, 648518346426145069, 648518346426240557, 648518346426242605, 648518346426242861, 648518346426243373, 648518346426243629, 648518346426243885, 648518346426244653, 648518346426244909, 648518346426246189, 648518346426439551, 648518346426444968, 648518346426474554, 648518346426531373, 648518346426548781, 648518346426886455, 648518346426984208, 648518346426995984, 648518346427038149, 648518346427047365, 648518346427050437, 648518346427050949, 648518346427051565, 648518346427052077, 648518346427053509, 648518346427053613, 648518346427053765, 648518346427054637, 648518346427072709, 648518346427077677, 648518346427078445, 648518346427090477, 648518346427092269, 648518346427092933, 648518346427093805, 648518346427094061, 648518346427094829, 648518346427096005, 648518346427112237, 648518346427133485, 648518346427140909, 648518346427143981, 648518346427199789, 648518346427274527, 648518346427282463, 648518346427289375, 648518346427291911, 648518346427311879, 648518346427320863, 648518346427342111, 648518346427347999, 648518346427359131, 648518346427362335, 648518346427396131, 648518346427401503, 648518346427418403, 648518346427419939, 648518346427421983, 648518346427436319, 648518346427492127, 648518346427494431, 648518346427500319, 648518346427500575, 648518346427501087, 648518346427504927, 648518346427506516, 648518346427525663, 648518346427550751, 648518346427560223, 648518346427560735, 648518346427560991, 648518346427561247, 648518346427566623, 648518346427569695, 648518346427570207, 648518346427577119, 648518346427600899, 648518346427746184, 648518346427752840, 648518346427758216, 648518346427758472, 648518346427761288, 648518346427765128, 648518346427765640, 648518346427765896, 648518346427766408, 648518346427767176, 648518346427776904, 648518346427779976, 648518346427780232, 648518346428376375, 648518346428396607, 648518346428534125, 648518346428534893, 648518346428535405, 648518346428535661, 648518346428563565, 648518346428569709, 648518346428907776, 648518346428908800, 648518346429046175, 648518346429189839, 648518346429315745, 648518346429337505, 648518346429340321, 648518346429340577, 648518346429354657, 648518346429355169, 648518346429378977, 648518346429383626, 648518346429384394, 648518346429388234, 648518346429388490, 648518346429395146, 648518346429398754, 648518346429422818, 648518346429423074, 648518346429435338, 648518346429447650, 648518346429451490, 648518346429656461, 648518346429729877, 648518346429734229, 648518346429737045, 648518346429740629, 648518346429770325, 648518346429804117, 648518346429804373, 648518346429804885, 648518346429807205, 648518346429908064, 648518346430055997, 648518346430218302, 648518346430219326, 648518346430219582, 648518346430222654, 648518346430223422, 648518346430224190, 648518346430224446, 648518346430224702, 648518346430224958, 648518346430225726, 648518346430225982, 648518346430228542, 648518346430229310, 648518346430230078, 648518346430230846, 648518346430231102, 648518346430233406, 648518346430233662, 648518346430233918, 648518346430234174, 648518346430235454, 648518346430235966, 648518346430238014, 648518346430238270, 648518346430239038, 648518346430251070, 648518346430251326, 648518346430251838, 648518346430252350, 648518346430252606, 648518346430253374, 648518346430310974, 648518346430311486, 648518346430311742, 648518346430313278, 648518346430313534, 648518346430314046, 648518346430460751, 648518346430647840, 648518346430655264, 648518346430728413, 648518346430795997, 648518346430873565, 648518346430995165, 648518346430995421, 648518346431013853, 648518346431059397, 648518346431059653, 648518346431060677, 648518346431061189, 648518346431061445, 648518346431061957, 648518346431062469, 648518346431062725, 648518346431063237, 648518346431064005, 648518346431064773, 648518346431066309, 648518346431067845, 648518346431068613, 648518346431068869, 648518346431069381, 648518346431069637, 648518346431069893, 648518346431070405, 648518346431071173, 648518346431071685, 648518346431072709, 648518346431072965, 648518346431073221, 648518346431073477, 648518346431073733, 648518346431076037, 648518346431076293, 648518346431090885, 648518346431332566, 648518346431332822, 648518346431334358, 648518346431334614, 648518346431334870, 648518346431335126, 648518346431335382, 648518346431335894, 648518346431336918, 648518346431400569, 648518346431467641, 648518346431517986, 648518346431521780, 648518346431582242, 648518346431702143, 648518346431830884, 648518346431831652, 648518346432346287, 648518346432474287, 648518346432519613, 648518346432583129, 648518346432598686, 648518346432599966, 648518346432602270, 648518346432616827, 648518346432626590, 648518346432629406, 648518346432629918, 648518346432630174, 648518346432693034, 648518346432883303, 648518346432924795, 648518346433008417, 648518346433281057, 648518346433371825, 648518346433568999, 648518346433638689, 648518346433778919, 648518346433780455, 648518346433824999, 648518346433825511, 648518346434138927, 648518346434148911, 648518346434439791, 648518346434588011, 648518346434588779, 648518346434592619, 648518346434601067, 648518346434603371, 648518346434606699, 648518346434606955, 648518346434607211, 648518346434616939, 648518346434617707, 648518346434658337, 648518346435005761, 648518346435413116, 648518346435539586, 648518346435650746, 648518346435878330, 648518346436187162, 648518346436514401, 648518346436635782, 648518346436675206, 648518346436728966, 648518346436789894, 648518346436790150, 648518346437082298, 648518346437116858, 648518346437192170, 648518346437192426, 648518346437192938, 648518346437193194, 648518346437221354, 648518346437222378, 648518346437278698, 648518346437280234, 648518346437280746, 648518346437281258, 648518346437339526, 648518346437345926, 648518346437351558, 648518346437370502, 648518346437398406, 648518346437398918, 648518346437411462, 648518346437411718, 648518346437411974, 648518346437433834, 648518346437437162, 648518346437438954, 648518346437498523, 648518346437518570, 648518346437519594, 648518346437520618, 648518346437522922, 648518346437523946, 648518346437524202, 648518346437524458, 648518346437524970, 648518346437525482, 648518346437525738, 648518346437527274, 648518346437528298, 648518346437528810, 648518346437529066, 648518346437529322, 648518346437530602, 648518346437531114, 648518346437531882, 648518346437532394, 648518346437532650, 648518346437532906, 648518346437533162, 648518346437533418, 648518346437533674, 648518346437534186, 648518346437534442, 648518346437534698, 648518346437534954, 648518346437535722, 648518346437535978, 648518346437536234, 648518346437536490, 648518346437536746, 648518346437537002, 648518346437537258, 648518346437538026, 648518346437538282, 648518346437538538, 648518346437538794, 648518346437539050, 648518346437581091, 648518346437640436, 648518346437645979, 648518346437670691, 648518346437671715, 648518346437680163, 648518346437680419, 648518346437686563, 648518346437688611, 648518346437688867, 648518346437693475, 648518346437699875, 648518346437700387, 648518346437702179, 648518346437702634, 648518346437702890, 648518346437726886, 648518346437774235, 648518346437775259, 648518346437775771, 648518346437805603, 648518346437831146, 648518346437831402, 648518346437838570, 648518346437839338, 648518346437841898, 648518346437842410, 648518346437842666, 648518346437847018, 648518346437848810, 648518346437865194, 648518346437866474, 648518346437870314, 648518346437874666, 648518346437875434, 648518346437875946, 648518346437907946, 648518346437908714, 648518346437910762, 648518346437924330, 648518346437928938, 648518346437929194, 648518346438023658, 648518346438399523, 648518346438399779, 648518346438400547, 648518346438402851, 648518346438404131, 648518346438404387, 648518346438405667, 648518346438410275, 648518346438421283, 648518346438425379, 648518346438426915, 648518346438428451, 648518346438428707, 648518346438430243, 648518346438446371, 648518346438447651, 648518346438468643, 648518346438641718, 648518346438641974, 648518346438668086, 648518346438670646, 648518346438754614, 648518346438757430, 648518346438868515, 648518346438906659, 648518346438906915, 648518346438949336, 648518346439538765, 648518346439644059, 648518346439944693, 648518346440307915, 648518346440399585, 648518346440596207, 648518346441099253, 648518346441572704, 648518346441651552, 648518346441726304, 648518346442388039, 648518346444260584, 648518346444261096, 648518346444261608, 648518346444263144, 648518346444483560, 648518346444750824, 648518346444751336, 648518346444754152, 648518346444754408, 648518346444800749, 648518346446482236, 648518346446864710, 648518346446867782, 648518346446872134, 648518346446923846, 648518346447076422, 648518346447082822, 648518346447083078, 648518346447083846, 648518346447084102, 648518346447084358, 648518346447084870, 648518346447085126, 648518346447085894, 648518346447086662, 648518346447087174, 648518346447088198, 648518346447088710, 648518346447093574, 648518346447107971, 648518346447141507, 648518346447188038, 648518346447304006, 648518346447304518, 648518346447306566, 648518346447307590, 648518346447315270, 648518346447315526, 648518346447316038, 648518346447316294, 648518346447316550, 648518346447317062, 648518346447317830, 648518346447320902, 648518346447323206, 648518346447323718, 648518346447323974, 648518346447328838, 648518346447329094, 648518346447329606, 648518346447329862, 648518346447330118, 648518346447330630, 648518346447371334, 648518346447386438, 648518346447387974, 648518346447394374, 648518346447398470, 648518346447402566, 648518346447402822, 648518346447404358, 648518346447408454, 648518346447410502, 648518346447673414, 648518346447702342, 648518346447702598, 648518346447702854, 648518346447703110, 648518346448017990, 648518346448164422, 648518346448959225, 648518346448961785, 648518346448982777, 648518346449522425, 648518346449804793, 648518346449805305, 648518346449805561, 648518346449806585, 648518346449807353, 648518346449807609, 648518346449818105, 648518346450081529, 648518346450916277, 648518346451782325, 648518346454229123]

materialization_version = 195
dec_prcnt = 95
overwrite_existing = False
show_full_path = False

segmentation_source_gs = "precomputed://gs://zheng_mouse_hippocampus_production/v2/seg_m195/"
segmentation_source_https = "precomputed://https://storage.googleapis.com/zheng_mouse_hippocampus_production/v2/seg_m195/"

print(f"root_ids: {[str(root_id) for root_id in root_ids]}")
print(f"materialization_version: {materialization_version}")
print(f"dec_prcnt: {dec_prcnt}")
print(f"overwrite_existing: {overwrite_existing}")

root_ids: ['648518346458354357']
materialization_version: 195
dec_prcnt: 95
overwrite_existing: False


## Environment Diagnostic

Check required packages in the existing environment. Do not install anything here.

In [ ]:
# import importlib
# from importlib import metadata

# required_packages = {
#     "meshparty": "meshparty",
#     "cloudvolume": "cloud-volume",
#     "h5py": "h5py",
#     "numpy": "numpy",
#     "pyvista": "pyvista",
#     "vtk": "vtk",
#     "pandas": "pandas",
#     "caveclient": "caveclient",
#     "dotenv": "python-dotenv",
#     "psutil": "psutil",
# }

# missing_packages = []
# for import_name, distribution_name in required_packages.items():
#     try:
#         importlib.import_module(import_name)
#         try:
#             version = metadata.version(distribution_name)
#         except metadata.PackageNotFoundError:
#             version = "installed, version unavailable"
#         print(f"{import_name}: {version}")
#     except ImportError:
#         missing_packages.append(import_name)
#         print(f"{import_name}: MISSING")

# if missing_packages:
#     raise RuntimeError(
#         "Required package(s) missing from this environment: "
#         + ", ".join(missing_packages)
#         + ". Install them in the notebook kernel environment before continuing."
#     )

## Shared Imports

Import the same libraries used by the proven single-root notebook, plus pandas for the final status table.

In [3]:
import gc
import sys
import time
import traceback
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import pyvista as pv
from caveclient import CAVEclient
from cloudvolume import CloudVolume
from meshparty import trimesh_io

import trimesh


def discover_project_root(start=Path.cwd().resolve()):
    for candidate in [start, *start.parents]:
        if (candidate / "notebooks" / "helpers").is_dir():
            return candidate
    raise RuntimeError(
        "Could not find the Pyr project root. Expected a parent directory containing "
        "notebooks/helpers."
    )


project_root = discover_project_root()
helpers_dir = project_root / "notebooks" / "helpers"
mesh_dir = project_root / "data" / "meshes"
dec_mesh_dir = mesh_dir / "dec"

if str(helpers_dir) not in sys.path:
    sys.path.insert(0, str(helpers_dir))

from cave_auth import load_cave_token
from path_behavior import format_path, print_path

mesh_dir.mkdir(parents=True, exist_ok=True)
dec_mesh_dir.mkdir(parents=True, exist_ok=True)

print_path("mesh_dir", mesh_dir, project_root, show_full_path)
print_path("dec_mesh_dir", dec_mesh_dir, project_root, show_full_path)


mesh_dir: data\meshes
dec_mesh_dir: data\meshes\dec


## Shared Pyr Segmentation Setup

Initialize the public Pyr `seg_m195` MeshParty source once as the primary source. The CAVE-advertised Graphene MeshParty source is initialized lazily only if a flat-source missing-manifest fallback is needed.


In [4]:
def make_mesh_meta_for_source(source):
    mm_candidate = trimesh_io.MeshMeta(
        cv_path=source,
        disk_cache_path=None,
        cache_size=1,
        map_gs_to_https=True,
    )
    cv = mm_candidate.cv
    info = cv.info
    mesh_source = cv.mesh
    return mm_candidate, info, mesh_source


def make_graphene_mesh_meta_for_source(source):
    return trimesh_io.MeshMeta(
        cv_path=source,
        disk_cache_path=None,
        cache_size=1,
    )


candidate_sources = [segmentation_source_gs, segmentation_source_https]
access_errors = []

mm = None
segmentation_source = None
segmentation_info = None

for candidate_source in candidate_sources:
    try:
        mm_candidate, info_candidate, mesh_source_candidate = make_mesh_meta_for_source(candidate_source)
        mm = mm_candidate
        segmentation_source = candidate_source
        segmentation_info = info_candidate
        print("Segmentation metadata access succeeded.")
        print(f"final segmentation source: {segmentation_source}")
        print(f"mesh source type: {type(mesh_source_candidate).__name__}")
        print(f"info type: {segmentation_info.get('type')}")
        print(f"mesh entry: {segmentation_info.get('mesh')}")
        break
    except Exception as exc:
        access_errors.append((candidate_source, type(exc).__name__, str(exc)))
        print(f"Segmentation metadata access failed for {candidate_source}")
        print(f"{type(exc).__name__}: {exc}")

if mm is None:
    raise RuntimeError(f"Could not access public seg_m195 metadata with tested sources: {access_errors}")

for forbidden in ["graphene://", "middleauth+", "minnie.microns-daf.com"]:
    assert forbidden not in segmentation_source.lower()
assert "seg_m195" in segmentation_source


graphene_source = None
mm_graphene = None


def get_graphene_mesh_meta():
    global graphene_source, mm_graphene
    if mm_graphene is not None:
        return mm_graphene

    cave_token, cave_token_source = load_cave_token(project_root)

    if cave_token:
        cave_client = CAVEclient("zheng_ca3", auth_token=cave_token)
    else:
        cave_client = CAVEclient("zheng_ca3")

    graphene_source = cave_client.info.segmentation_source(format_for="cloudvolume")
    if not isinstance(graphene_source, str) or not graphene_source.startswith("graphene://"):
        raise RuntimeError(f"CAVE returned a non-Graphene segmentation source: {graphene_source}")

    mm_graphene = make_graphene_mesh_meta_for_source(graphene_source)
    print("Graphene metadata access succeeded.")
    print(f"Graphene segmentation source: {graphene_source}")
    print(f"Graphene mesh source type: {type(mm_graphene.cv.mesh).__name__}")
    return mm_graphene


Segmentation metadata access succeeded.
final segmentation source: precomputed://gs://zheng_mouse_hippocampus_production/v2/seg_m195/
mesh source type: ShardedMultiLevelPrecomputedMeshSource
info type: segmentation
mesh entry: mesh_mip_1_err_40


## Helper Functions

These helpers keep each root ID isolated while preserving the proven filename conventions, H5 writer/reader, and PyVista triangular-face conversion.

In [5]:
def paths_for_root_id(root_id):
    root_id_str = str(root_id)
    h5_path = mesh_dir / f"mesh_{root_id_str}_mat{materialization_version}.h5"
    ply_path = dec_mesh_dir / f"mesh_{root_id_str}_mat{materialization_version}_dec{dec_prcnt}.ply"
    return h5_path, ply_path


def validate_mesh_arrays(vertices, faces):
    vertices = np.asarray(vertices)
    faces = np.asarray(faces)
    assert vertices.ndim == 2 and vertices.shape[1] == 3
    assert faces.ndim == 2 and faces.shape[1] == 3
    assert len(vertices) > 0
    assert len(faces) > 0
    return vertices, faces


def read_h5_file(file_path):
    with h5py.File(file_path, "r") as f:
        vertices = np.array(f["vertices"])
        faces = np.array(f["faces"])
    return validate_mesh_arrays(vertices, faces)


def create_pyvista_mesh(vertices, faces):
    face_sizes = np.full((faces.shape[0], 1), 3, dtype=faces.dtype)
    faces_with_size = np.hstack([face_sizes, faces]).ravel()
    return pv.PolyData(vertices, faces_with_size)


def validate_pyvista_mesh(mesh):
    assert mesh.n_points > 0
    assert mesh.n_cells > 0
    points = np.asarray(mesh.points)
    assert points.ndim == 2 and points.shape[1] == 3
    return mesh


def temp_path_for(file_path):
    return file_path.with_name(f"{file_path.stem}.tmp{file_path.suffix}")


def remove_if_exists(file_path):
    if file_path.exists():
        file_path.unlink()


def write_h5_atomic(final_path, vertices, faces, mesh):
    temp_path = temp_path_for(final_path)
    try:
        remove_if_exists(temp_path)
        trimesh_io.write_mesh_h5(
            temp_path,
            vertices=vertices,
            faces=faces,
            normals=getattr(mesh, "face_normals", None),
            link_edges=getattr(mesh, "link_edges", None),
            node_mask=getattr(mesh, "node_mask", None),
            overwrite=True,
        )
        assert temp_path.exists()
        reloaded_vertices, reloaded_faces = read_h5_file(temp_path)
        assert len(reloaded_vertices) == len(vertices)
        assert len(reloaded_faces) == len(faces)
        temp_path.replace(final_path)
        assert final_path.exists()
    except Exception:
        remove_if_exists(temp_path)
        raise


def save_ply_atomic(final_path, mesh):
    temp_path = temp_path_for(final_path)
    try:
        remove_if_exists(temp_path)
        mesh.save(temp_path)
        assert temp_path.exists()
        reloaded_mesh = validate_pyvista_mesh(pv.read(temp_path))
        assert reloaded_mesh.n_points == mesh.n_points
        assert reloaded_mesh.n_cells == mesh.n_cells
        temp_path.replace(final_path)
        assert final_path.exists()
        return reloaded_mesh
    except Exception:
        remove_if_exists(temp_path)
        raise


def is_manifest_not_found_error(exc):
    return type(exc).__name__ == "MeshDecodeError" and "Manifest not found" in str(exc)


def classify_exception(exc):
    error_type = type(exc).__name__
    error_message = str(exc)
    if is_manifest_not_found_error(exc):
        return "missing_manifest", error_type, error_message
    return "failed", error_type, error_message


## Per-Root Unit Of Work

Process one root ID at a time. A failure for one root ID is captured in that root's status record and does not stop later root IDs.

In [6]:
def process_root_id(root_id):
    root_id_str = str(root_id)
    h5_path, ply_path = paths_for_root_id(root_id_str)
    start_time = time.time()
    mesh = None
    vertices = None
    faces = None
    reloaded_decimated_mesh = None
    pv_mesh = None
    decimated_mesh = None

    result = {
        "root_id": root_id_str,
        "status": None,
        "materialization_version": materialization_version,
        "mesh_source": None,
        "h5_status": None,
        "ply_status": None,
        "original_vertices": None,
        "original_faces": None,
        "decimated_vertices": None,
        "decimated_faces": None,
        "elapsed_seconds": None,
        "h5_path": str(h5_path),
        "ply_path": str(ply_path),
        "flat_error_type": "",
        "flat_error_message": "",
        "graphene_error_type": "",
        "graphene_error_message": "",
        "error_type": "",
        "error_message": "",
        "error": "",
    }

    try:
        print(f"\nProcessing root ID: {root_id_str}")

        if ply_path.exists() and not overwrite_existing:
            try:
                reloaded_decimated_mesh = validate_pyvista_mesh(pv.read(ply_path))
                result["decimated_vertices"] = reloaded_decimated_mesh.n_points
                result["decimated_faces"] = reloaded_decimated_mesh.n_cells
                result["status"] = "success"
                result["mesh_source"] = "existing_ply"
                result["h5_status"] = "skipped valid PLY"
                result["ply_status"] = "reused existing PLY"
                print(f"Valid PLY already exists; skipping root ID {root_id_str}: {format_path(ply_path, project_root, show_full_path)}")
                return result
            except Exception as exc:
                print(f"Warning: invalid existing PLY for root ID {root_id_str}; removing {format_path(ply_path, project_root, show_full_path)}: {type(exc).__name__}: {exc}")
                ply_path.unlink()

        if h5_path.exists() and not overwrite_existing:
            try:
                print(f"Reusing existing H5: {format_path(h5_path, project_root, show_full_path)}")
                vertices, faces = read_h5_file(h5_path)
                result["mesh_source"] = "existing_h5"
                result["h5_status"] = "reused existing H5"
            except Exception as exc:
                print(f"Warning: invalid existing H5 for root ID {root_id_str}; removing {format_path(h5_path, project_root, show_full_path)}: {type(exc).__name__}: {exc}")
                h5_path.unlink()
                result["h5_status"] = None

        if result["h5_status"] != "reused existing H5":
            try:
                mesh = mm.mesh(
                    seg_id=int(root_id_str),
                    remove_duplicate_vertices=True,
                    cache_mesh=True,
                )
                result["mesh_source"] = "seg_m195"
            except Exception as flat_exc:
                result["flat_error_type"] = type(flat_exc).__name__
                result["flat_error_message"] = str(flat_exc)
                if not is_manifest_not_found_error(flat_exc):
                    raise

                print(
                    f"Flat seg_m195 manifest not found for root ID {root_id_str}; "
                    "trying Graphene fallback"
                )
                try:
                    graphene_mm = get_graphene_mesh_meta()
                    mesh = graphene_mm.mesh(
                        seg_id=int(root_id_str),
                        remove_duplicate_vertices=True,
                    )
                    result["mesh_source"] = "graphene_fallback"
                    print(f"Graphene fallback recovered root ID {root_id_str}")
                except Exception as graphene_exc:
                    result["status"] = "missing_manifest"
                    result["mesh_source"] = "unavailable"
                    result["graphene_error_type"] = type(graphene_exc).__name__
                    result["graphene_error_message"] = str(graphene_exc)
                    result["error_type"] = type(graphene_exc).__name__
                    result["error_message"] = str(graphene_exc)
                    result["error"] = (
                        f"Flat MeshDecodeError: {result['flat_error_message']}; "
                        f"Graphene {type(graphene_exc).__name__}: {graphene_exc}"
                    )
                    result["h5_status"] = result["h5_status"] or "failed"
                    result["ply_status"] = result["ply_status"] or "failed"
                    print(f"Failed root ID {root_id_str}: {result['error']}")
                    return result

            vertices, faces = validate_mesh_arrays(mesh.vertices, mesh.faces)

            write_h5_atomic(h5_path, vertices, faces, mesh)
            result["h5_status"] = "downloaded"
            print(f"Downloaded and saved H5: {format_path(h5_path, project_root, show_full_path)}")

        result["original_vertices"] = len(vertices)
        result["original_faces"] = len(faces)

        pv_mesh = create_pyvista_mesh(vertices, faces)
        assert pv_mesh.n_points == len(vertices)
        assert pv_mesh.n_cells == len(faces)

        target_reduction = dec_prcnt / 100
        decimated_mesh = validate_pyvista_mesh(pv_mesh.decimate(target_reduction=target_reduction))
        result["decimated_vertices"] = decimated_mesh.n_points
        result["decimated_faces"] = decimated_mesh.n_cells

        reloaded_decimated_mesh = save_ply_atomic(ply_path, decimated_mesh)
        assert reloaded_decimated_mesh.n_points == result["decimated_vertices"]
        assert reloaded_decimated_mesh.n_cells == result["decimated_faces"]
        result["status"] = "success"
        result["ply_status"] = "decimated"
        print(f"Decimated and saved PLY: {format_path(ply_path, project_root, show_full_path)}")

    except Exception as exc:
        status, error_type, error_message = classify_exception(exc)
        result["status"] = status
        result["mesh_source"] = result["mesh_source"] or ("unavailable" if status == "missing_manifest" else "failed")
        result["error_type"] = error_type
        result["error_message"] = error_message
        result["error"] = f"{error_type}: {error_message}"
        result["h5_status"] = result["h5_status"] or "failed"
        result["ply_status"] = result["ply_status"] or "failed"
        print(f"Failed root ID {root_id_str}: {result['error']}")
    finally:
        mesh = None
        vertices = None
        faces = None
        reloaded_decimated_mesh = None
        pv_mesh = None
        decimated_mesh = None
        gc.collect()
        result["elapsed_seconds"] = time.time() - start_time

    return result

In [7]:
import psutil
import os

process = psutil.Process(os.getpid())

print(f"Process RAM: {process.memory_info().rss / 1024**3:.2f} GB")
print(f"System available RAM: {psutil.virtual_memory().available / 1024**3:.2f} GB")

Process RAM: 0.36 GB
System available RAM: 3.01 GB


## Run Batch

Process each root ID independently and collect one structured status record per root.

In [8]:
results = []
results_csv_path = mesh_dir / "mesh_batch_results.csv"

for root_id in root_ids:
    results.append(process_root_id(root_id))
    results_df = pd.DataFrame(results)
    results_df.to_csv(results_csv_path, index=False)
    print(f"Saved current results table: {format_path(results_csv_path, project_root, show_full_path)}")

results_df = pd.DataFrame(results)
results_df.to_csv(results_csv_path, index=False)
results_display_df = results_df.copy()
for path_column in ["h5_path", "ply_path"]:
    if path_column in results_display_df.columns:
        results_display_df[path_column] = results_display_df[path_column].map(
            lambda path: format_path(path, project_root, show_full_path)
        )
display(results_display_df)


Processing root ID: 648518346458354357


100%|██████████| 1/1 [00:04<00:00,  4.82s/it]


Downloaded and saved H5: data\meshes\mesh_648518346458354357_mat195.h5
Decimated and saved PLY: data\meshes\dec\mesh_648518346458354357_mat195_dec95.ply
Saved current results table: data\meshes\mesh_batch_results.csv


,root_id,status,materialization_version,mesh_source,h5_status,ply_status,original_vertices,original_faces,decimated_vertices,decimated_faces,elapsed_seconds,h5_path,ply_path,flat_error_type,flat_error_message,graphene_error_type,graphene_error_message,error_type,error_message,error
0,648518346458354357,success,195,seg_m195,downloaded,decimated,639986,1271465,32048,63572,17.871969,data\meshes\mesh_648518346458354357_mat195.h5,data\meshes\dec\mesh_648518346458354357_mat195...,,,,,,,


## Batch Summary

Summarize requested, successful, failed, reused, downloaded, and newly decimated roots.

In [9]:
def final_ply_is_valid(root_id):
    _, ply_path = paths_for_root_id(root_id)
    if not ply_path.exists():
        return False
    try:
        validate_pyvista_mesh(pv.read(ply_path))
        return True
    except Exception:
        return False


def print_python_int_list(name, values):
    print(f"{name} = {[int(value) for value in values]}")


successful_mesh_ids = [int(root_id) for root_id in root_ids if final_ply_is_valid(root_id)]
missing_manifest_ids = [
    int(root_id)
    for root_id in results_df.loc[results_df["status"].eq("missing_manifest"), "root_id"]
]
successful_mesh_id_set = set(successful_mesh_ids)
missing_manifest_id_set = set(missing_manifest_ids)
failed_mesh_ids = [
    int(root_id)
    for root_id in root_ids
    if int(root_id) not in successful_mesh_id_set
    and int(root_id) not in missing_manifest_id_set
]

total_requested = len(root_ids)
failed = len(missing_manifest_ids) + len(failed_mesh_ids)
successful = len(successful_mesh_ids)
reused_existing_files = int(
    (results_df["h5_status"].eq("reused existing H5")).sum()
    + (results_df["ply_status"].eq("reused existing PLY")).sum()
)
newly_downloaded = int(results_df["h5_status"].eq("downloaded").sum())
newly_decimated = int(results_df["ply_status"].eq("decimated").sum())
if "mesh_source" not in results_df.columns:
    results_df["mesh_source"] = ""
newly_retrieved_via_seg_m195 = int(
    (results_df["status"].eq("success") & results_df["mesh_source"].eq("seg_m195")).sum()
)
recovered_via_graphene_fallback = int(
    (results_df["status"].eq("success") & results_df["mesh_source"].eq("graphene_fallback")).sum()
)
reused_existing_ply = int(
    (results_df["status"].eq("success") & results_df["mesh_source"].eq("existing_ply")).sum()
)
reused_existing_h5 = int(
    (results_df["status"].eq("success") & results_df["mesh_source"].eq("existing_h5")).sum()
)
remaining_missing_unavailable = len(missing_manifest_ids)
other_failures = len(failed_mesh_ids)

print(f"requested IDs: {total_requested}")
print(f"successful valid PLYs: {successful}")
print(f"missing manifests after fallback: {len(missing_manifest_ids)}")
print(f"other failures: {len(failed_mesh_ids)}")
print(f"reused existing files: {reused_existing_files}")
print(f"newly downloaded: {newly_downloaded}")
print(f"newly decimated: {newly_decimated}")
print()
print("source recovery summary:")
print(f"newly retrieved via seg_m195: {newly_retrieved_via_seg_m195}")
print(f"recovered via Graphene fallback: {recovered_via_graphene_fallback}")
print(f"reused existing PLY: {reused_existing_ply}")
print(f"reused existing H5: {reused_existing_h5}")
print(f"remaining unavailable/failures: {remaining_missing_unavailable + other_failures}")
print(f"remaining missing/unavailable: {remaining_missing_unavailable}")
print(f"other failures: {other_failures}")
print()
print_python_int_list("successful_mesh_ids", successful_mesh_ids)
print()
print_python_int_list("missing_manifest_ids", missing_manifest_ids)
print()
print_python_int_list("failed_mesh_ids", failed_mesh_ids)


requested IDs: 1
successful valid PLYs: 1
missing manifests after fallback: 0
other failures: 0
reused existing files: 0
newly downloaded: 1
newly decimated: 1

source recovery summary:
newly retrieved via seg_m195: 1
recovered via Graphene fallback: 0
reused existing PLY: 0
reused existing H5: 0
remaining unavailable/failures: 0
remaining missing/unavailable: 0
other failures: 0

successful_mesh_ids = [648518346458354357]

missing_manifest_ids = []

failed_mesh_ids = []
